In [1]:
import torch
import transformer_lens
from sae_lens import SAE

C:\Users\supervisor.LTCPC-280426-01\tlab\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",  # Имя датасета SAE
    sae_id="blocks.6.hook_resid_pre",  # Какой слой
    device="cuda"
)

C:\Users\supervisor.LTCPC-280426-01\tlab\lib\site-packages\sae_lens\saes\sae.py:254: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


In [4]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small",  device=device, use_cache=True)
# out = model.generate(
#     'I see a picture on the wall; the picture depicts',
#     max_new_tokens=50,
#     temperature=0.85,
#     top_p=1.0,
#     verbose=False
# )
# print(f"{out}")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 16443.02it/s]


Loaded pretrained model gpt2-small into HookedTransformer


In [5]:
import json

with open('deepseek.json') as f:
    data = json.load(f)
print(len(data))

997


In [6]:
data = data[:300]

In [7]:
idx = [9191]
alphas = [0.0 , 1.0, 3.0, 5.0, 7.0, 9.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0]
v = sae.W_dec[idx]
alpha = 10


In [8]:
print(v.mean(), v.std())

tensor(-0.0018, device='cuda:0', grad_fn=<MeanBackward0>) tensor(0.0361, device='cuda:0', grad_fn=<StdBackward0>)


In [9]:
activates = []
post_activates = []
def hook_steering(tensor, hook):
    # print(tensor.shape)
    # if len(activates) > 0:
    #     activates.append(torch.concat([activates[-1], tensor], dim=1))
    # else:
    activates.append(tensor)
    tensor = tensor + alpha * v
    post_activates.append(tensor)
    return tensor

In [ ]:
res_generator = []
for alpha in alphas:
    for promt in data:

        model.add_hook(
            name="blocks.6.hook_resid_pre",
            hook=hook_steering,
            dir="fwd"
        )

        logits = model.generate(promt, max_new_tokens=100, temperature=0.8)

        res_generator.append({
            'alpha': alpha,
            'promt': promt,
            'continious': logits,
        })
        model.reset_hooks()

 96%|█████████▌| 96/100 [00:02<00:00, 47.14it/s]

In [ ]:
res_generator[0]

In [1]:
import json

alphas = [0.0, 5.0, 10.0,11.0,12.0,13.0,14.0,15.0, 20.0]

idx = [0,1,2,3,4,5,6,7,8]
prompts = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]
data = []
for prompt in prompts:
    for alpha in alphas:
        with open(f'res_with_score/deepseek/{prompt}_{alpha}_0.json', 'w', encoding='utf-8') as file:
            json.dump(data, file)
